In [0]:
# All data needed for dashboard
from pyspark.sql import functions as F

# Query 1 - State risk overview
state_risk = spark.table(
    "ecommerce.gold.state_predictions_gold"
).select(
    "region",
    "avg_unemployment_rate",
    "skill_gap_score",
    "risk_level",
    "intervention_priority",
    "prediction"
).orderBy(F.desc("avg_unemployment_rate"))

state_risk.write.format("delta").mode("overwrite") \
    .saveAsTable("ecommerce.gold.dashboard_state_risk")

# Query 2 - Region summary
region_summary = spark.table(
    "ecommerce.gold.state_predictions_gold"
).groupBy("region").agg(
    F.avg("avg_unemployment_rate")
     .alias("region_avg_unemployment"),
    F.count("*").alias("total_states"),
    F.sum(F.when(F.col("prediction") == 1, 1)
           .otherwise(0)).alias("states_need_intervention"),
    F.max("avg_unemployment_rate")
     .alias("highest_unemployment")
).orderBy(F.desc("region_avg_unemployment"))

region_summary.write.format("delta").mode("overwrite") \
    .saveAsTable("ecommerce.gold.dashboard_region_summary")

# Query 3 - Job category demand
job_demand = spark.table(
    "ecommerce.gold.live_demand_gold"
).orderBy(F.desc("live_job_count"))

# Query 4 - Experience level gaps
exp_gap = spark.table(
    "ecommerce.silver.exp_demand_silver"
).orderBy(F.desc("jobs_count"))

exp_gap.write.format("delta").mode("overwrite") \
    .saveAsTable("ecommerce.gold.dashboard_exp_gap")

# Query 5 - Top roles
top_roles = spark.table(
    "ecommerce.silver.top_roles_silver"
).orderBy(F.desc("avg_salary_usd")).limit(20)

top_roles.write.format("delta").mode("overwrite") \
    .saveAsTable("ecommerce.gold.dashboard_top_roles")

print("All dashboard tables ready ✓")

All dashboard tables ready ✓
